# 02 — Build prediction features and labels

Each sample is one observed station/time pair. Features use only the current reading and exact elapsed-time history at or before that time. The target sums the next six five-minute readings; incomplete windows are excluded from supervised evaluation.

In [ ]:
from datetime import datetime
from pathlib import Path
import json
import os
import math

from IPython.display import display
from pyspark.sql import SparkSession, Window, functions as F

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
BUCKET = os.environ.get("BUCKET", "sg-rainfall-dsa5208-261011-data")
RUN_ID = os.environ.get("RUN_ID", "20261012-full-01")
ROOT = f"gs://{BUCKET}/runs/{RUN_ID}"
config = json.loads((PROJECT_ROOT / "config" / "experiment.json").read_text())
spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "Asia/Singapore")
spark.conf.set("spark.sql.ansi.enabled", "false")

if not (config["interval_semantics_confirmed"] or config.get("interval_end_assumption_accepted", False)):
    raise ValueError("The project assumes timestamps mark the end of each five-minute interval.")


## Feature definitions

Variant A is a seasonal/threshold baseline feature set. Variant B adds current rainfall, exact-time lags and rolling summaries. Missing history remains explicit through lag flags and rolling coverage; a missing current reading is not a supported prediction query.

In [ ]:
BASE_FEATURES = ["hour_sin", "hour_cos", "day_sin", "day_cos", "threshold_mm"]


def feature_columns(config, variant="B"):
    columns = list(BASE_FEATURES)
    if variant == "A":
        return columns
    if variant != "B":
        raise ValueError("Only feature variants A and B are defined.")
    columns.append("rain_current")
    for minutes in config["lag_minutes"]:
        columns.extend([f"lag_{minutes}", f"lag_{minutes}_missing"])
    for minutes in config["rolling_minutes"]:
        columns.extend([
            f"sum_{minutes}", f"max_{minutes}", f"wet_count_{minutes}",
            f"count_{minutes}", f"coverage_{minutes}",
        ])
    return columns


def threshold_units(value, scale):
    scaled = int(round(float(value) * scale))
    if not math.isclose(scaled / scale, float(value), rel_tol=0, abs_tol=1e-10):
        raise ValueError("Threshold exceeds supported numeric precision.")
    return scaled


In [ ]:
def add_history_features(clean, config):
    by_station_time = Window.partitionBy("station_id").orderBy("epoch")
    frame = clean.withColumn("rain_current", F.col("rain_mm"))

    # A range window asks for the reading exactly N minutes earlier; it never
    # shifts by row count when a station has missing intervals.
    for minutes in config["lag_minutes"]:
        exact_lag = F.first("rain_mm", ignorenulls=True).over(
            by_station_time.rangeBetween(-minutes * 60, -minutes * 60)
        )
        frame = frame.withColumn(f"lag_{minutes}", exact_lag).withColumn(
            f"lag_{minutes}_missing", F.col(f"lag_{minutes}").isNull().cast("double")
        )

    for minutes in config["rolling_minutes"]:
        window = by_station_time.rangeBetween(-(minutes - 5) * 60, 0)
        frame = frame.withColumn(f"sum_{minutes}", F.sum("rain_mm").over(window))
        frame = frame.withColumn(f"max_{minutes}", F.max("rain_mm").over(window))
        frame = frame.withColumn(
            f"wet_count_{minutes}",
            F.sum(F.when(F.col("rain_mm") > 0, 1).otherwise(0)).over(window),
        )
        frame = frame.withColumn(f"count_{minutes}", F.count("rain_mm").over(window).cast("double"))
        frame = frame.withColumn(f"coverage_{minutes}", F.col(f"count_{minutes}") / (minutes / 5))
    return frame


In [ ]:
def add_calendar_features(frame):
    timestamp = F.to_timestamp(F.from_unixtime("epoch"))
    hour_fraction = F.hour(timestamp) + F.minute(timestamp) / 60.0
    start_of_year = F.make_date(F.year(timestamp), F.lit(1), F.lit(1))
    start_next_year = F.make_date(F.year(timestamp) + 1, F.lit(1), F.lit(1))
    year_fraction = (F.dayofyear(timestamp) - 1 + hour_fraction / 24) / F.datediff(
        start_next_year, start_of_year
    )
    angle = 2 * F.lit(math.pi)
    return (
        frame.withColumn("hour_sin", F.sin(angle * hour_fraction / 24))
        .withColumn("hour_cos", F.cos(angle * hour_fraction / 24))
        .withColumn("day_sin", F.sin(angle * year_fraction))
        .withColumn("day_cos", F.cos(angle * year_fraction))
        .withColumn("month", F.month(timestamp))
        .withColumn("hour", F.hour(timestamp))
        .withColumn("local_date", F.to_date(timestamp))
    )


def add_future_target(frame):
    future = Window.partitionBy("station_id").orderBy("epoch").rangeBetween(300, 1800)
    frame = frame.withColumn("future_units", F.sum("rain_units").over(future))
    frame = frame.withColumn("future_count", F.count("rain_units").over(future))
    frame = frame.withColumn(
        "future_min_epoch", F.min(F.when(F.col("rain_units").isNotNull(), F.col("epoch"))).over(future)
    )
    frame = frame.withColumn(
        "future_max_epoch", F.max(F.when(F.col("rain_units").isNotNull(), F.col("epoch"))).over(future)
    )
    return frame.withColumn(
        "label_valid",
        (F.col("future_count") == 6)
        & (F.col("future_min_epoch") == F.col("epoch") + 300)
        & (F.col("future_max_epoch") == F.col("epoch") + 1800),
    )


## Create the decision grid and target

The exact-grid observations define candidate prediction times. A target is valid only when all six future readings exist at the expected timestamps. Equality with a rainfall threshold is a negative label.

In [ ]:
def build_base_features(clean, config):
    frame = add_history_features(clean, config)
    frame = add_calendar_features(frame)
    frame = frame.withColumn("onset", F.coalesce(F.col("sum_30"), F.lit(0)) == 0)
    return add_future_target(frame)


def assign_splits(base, config):
    boundaries = [int(datetime.fromisoformat(value).timestamp()) for value in config["split_boundaries"]]
    split = F.lit(None).cast("string")
    for name, start, end in zip(config["split_names"], boundaries[:-1], boundaries[1:]):
        # Exclude any target window that touches the next calendar partition.
        split = F.when((F.col("epoch") >= start) & (F.col("epoch") + 1800 < end), F.lit(name)).otherwise(split)
    return base.withColumn("split", split)


def expand_thresholds(base, config, thresholds=None):
    values = thresholds if thresholds is not None else config["thresholds_mm"]
    rows = [(float(value), threshold_units(value, config["rainfall_integer_scale"])) for value in values]
    grid = base.sparkSession.createDataFrame(rows, "threshold_mm double, threshold_units long")
    expanded = base.crossJoin(F.broadcast(grid))
    if "future_units" in base.columns:
        expanded = expanded.withColumn(
            "label", (F.col("future_units") > F.col("threshold_units")).cast("double")
        )
    return expanded


In [ ]:
def feature_dictionary(config):
    descriptions = {
        "hour_sin": "cyclic hour at prediction time",
        "hour_cos": "cyclic hour at prediction time",
        "day_sin": "cyclic day of year",
        "day_cos": "cyclic day of year",
        "threshold_mm": "requested rainfall threshold",
        "rain_current": "observed rainfall at the prediction timestamp",
    }
    for minutes in config["lag_minutes"]:
        descriptions[f"lag_{minutes}"] = f"same-station reading exactly {minutes} minutes earlier"
        descriptions[f"lag_{minutes}_missing"] = f"indicator for a missing {minutes}-minute lag"
    for minutes in config["rolling_minutes"]:
        descriptions.update({
            f"sum_{minutes}": f"rainfall sum over the last {minutes} minutes",
            f"max_{minutes}": f"maximum reading over the last {minutes} minutes",
            f"wet_count_{minutes}": f"number of wet readings over the last {minutes} minutes",
            f"count_{minutes}": f"available readings over the last {minutes} minutes",
            f"coverage_{minutes}": f"available-reading fraction over the last {minutes} minutes",
        })
    return [{"name": name, "meaning": descriptions[name]} for name in feature_columns(config)]

display(feature_dictionary(config))


In [ ]:
with_history = spark.read.parquet(f"{ROOT}/cleaned")
base = assign_splits(build_base_features(with_history, config), config)
base.write.mode("overwrite").partitionBy("split").parquet(f"{ROOT}/base_features")

attrition = base.groupBy("split", "label_valid").agg(
    F.count("*").alias("base_samples"),
    F.min("epoch").alias("first_prediction_epoch"),
    F.max("epoch").alias("last_prediction_epoch"),
)
attrition.orderBy("split", "label_valid").show(20, truncate=False)
base.select(
    "station_id", "observation_time", "rain_current", "lag_5", "sum_30",
    "future_units", "future_count", "label_valid", "split",
).where("split = 'train'").show(10, truncate=False)
display({
    "decision_grid": "observed exact-grid station/time keys",
    "missing_current_queries": "unsupported; no model probability is returned",
    "future_window": "six readings at +5, +10, +15, +20, +25, +30 minutes",
    "base_feature_count": len(feature_columns(config, "B")),
})


In [ ]:
spark.stop()
print("Base features and valid-window labels saved.")
